# Segmentation

The [Quickstart](quickstart.ipynb) shrank the data by keeping fewer **periods** — a
year of days became a handful of typical days. **Segmentation** is a second,
independent lever that works *inside* each period.

A typical day still has 24 hourly values. Many of those hours are nearly
identical — overnight load barely moves for hours. Segmentation merges runs of
similar consecutive steps into a single **segment** with one value, so a 24-hour
day might be described by just 6 segments. Segments are **variable length**: long
where the profile is flat, short where it changes quickly.

It is easiest to just see it.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import SegmentConfig

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## See it: 24 hours → 6 segments

Aggregate to 6 typical days, but describe each day with **6 segments** instead of
24 hourly steps:

In [ ]:
result = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6),
)

Now compare the **original** hourly load with the **reconstructed** series built
from the segmented typical days. The reconstruction is a step function — each flat
step is one segment, held constant across the hours it covers. Where the original
is busy (the daily ramp), segments are short; where it is calm (overnight), one
long segment spans many hours.

Here is one week — each flat step is a segment. (The plot is interactive, so you can zoom further into a single day.)

In [ ]:
week = slice("2010-01-11", "2010-01-17")
result.plot.compare(columns=["Load"], time_slice=week, color="source")

That is the whole idea: replace 24 hourly values with a few representative steps,
chosen to follow the shape of the data — longer where it is flat, shorter where it
moves.

## Why bother: the budget

The two levers multiply. Six typical days at hourly resolution already cut the
six-week series down a lot; segmenting each day into 6 steps cuts it much further
— for a small accuracy cost.

In [ ]:
plain = tsam.aggregate(data, n_clusters=6, period_duration="1D")

original_steps = len(data)
plain_steps = plain.n_clusters * plain.n_timesteps_per_period  # 6 days x 24 h
seg_steps = result.n_clusters * result.n_segments  # 6 days x 6 segments

pd.DataFrame(
    {
        "time steps": [original_steps, plain_steps, seg_steps],
        "reduction": [
            "—",
            f"{1 - plain_steps / original_steps:.0%}",
            f"{1 - seg_steps / original_steps:.0%}",
        ],
        "mean RMSE": [
            0.0,
            round(float(plain.accuracy.rmse.mean()), 4),
            round(float(result.accuracy.rmse.mean()), 4),
        ],
    },
    index=["original (hourly)", "6 days x 24 h", "6 days x 6 segments"],
)

So you have two dials: how many typical periods, and how fine each one is.
[How small can you go?](tuning.ipynb) searches both for the best combination at a
target size.